# Computational Thermodynamics — Al-Zn Binary System (Mey 1993)

Interactive CALPHAD analysis using **pycalphad**.

> **No installation needed.** Press **Runtime → Run all** and scroll down to use the sliders.


In [ ]:
# @title Step 1 — Install packages (run once per Colab session)
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "pycalphad", "scipy", "-q"], check=True)
print("Packages ready!")


In [ ]:
# @title Step 2 — Imports and helper utilities
import io, warnings
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.interpolate import interp1d
warnings.filterwarnings("ignore")

from pycalphad import Database, binplot, calculate, equilibrium, variables as v
import ipywidgets as widgets
from IPython.display import display, clear_output, Image as IPImage

plt.rcParams.update({
    "font.size": 11, "axes.labelsize": 13, "axes.titlesize": 14,
    "lines.linewidth": 2.0, "figure.dpi": 120,
})

def fig2img(fig):
    """Convert a matplotlib figure to an IPython Image (PNG bytes)."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=120)
    buf.seek(0)
    plt.close(fig)
    return IPImage(buf.read())

print("Imports OK")


In [ ]:
# @title Step 3 — Load Al-Zn database (embedded, no file needed)
ALZN_TDB = """
$ ALZN
$
$ TDB-file for the thermodynamic assessment of the Al-ZN system
$
$-------------------------------------------------------------------------------
$ 2011.11.9
$ 
$ TDB file created by T.Abe, K.Hashimoto and Y.sawada
$
$ Particle Simulation and Thermodynamics Group, National Institute for 
$ Materials Science. 1-2-1 Sengen, Tsukuba, Ibaraki 305-0047, Japan
$ e-mail: abe.taichi@nims.go.jp
$ Copyright (C) NIMS 2009
$
$ ------------------------------------------------------------------------------
$ PARAMETERS ARE TAKEN FROM 
$ Reevaluation of the Al-Zn System,
$ Sabine an Mey, Z.Metallkd., 84 (1993) 451-455.
$
$ ------------------------------------------------------------------------------

 ELEMENT /-   ELECTRON_GAS              0.0000E+00  0.0000E+00  0.0000E+00!
 ELEMENT VA   VACUUM                    0.0000E+00  0.0000E+00  0.0000E+00!
 ELEMENT AL   FCC_A1                    2.6982E+01  4.5773E+03  2.8322E+01!
 ELEMENT ZN   HCP_ZN                    6.5390E+01  5.6568E+03  4.1631E+01!

$-------------------------------------------------------------------------------
$ FUNCTIONS FOR PURE AND OTHERS
$-------------------------------------------------------------------------------
 FUNCTION  GHSERAL  298.0
    -7976.15+137.0715*T-24.36720*T*LN(T)-1.884662E-3*T**2-0.877664E-6*T**3
    +74092*T**(-1);                                                   700.00 Y
    -11276.24+223.0269*T-38.58443*T*LN(T)+18.531982E-3*T**2-5.764227E-6*T**3
    +74092*T**(-1);                                                    933.6 Y
    -11277.68+188.6620*T-31.74819*T*LN(T)-1234.26E25*T**(-9);        2900.00 N ! 
 FUNCTION  GALLIQ   298.0
    +3029.403+125.2307*T-24.36720*T*LN(T)-1.884662E-3*T**2-0.877664E-6*T**3
    +74092*T**(-1)+79.401E-21*T**7;                                   700.00 Y
    -270.6860+211.1861*T-38.58443*T*LN(T)+18.53198E-3*T**2-5.764227E-6*T**3
    +74092*T**(-1)+79.401E-21*T**7;                                    933.6 Y
    -795.7090+177.4100*T-31.74819*T*LN(T);                           2900.00 N !
 FUNCTION  GALHCP   298.0  +5481-1.8*T+GHSERAL#;                      6000  N !


 FUNCTION GHSERZN    298.0  -7285.787+118.4693*T-23.70131*T*LN(T)
     -.001712034*T**2-1.264963E-06*T**3;                              692.7 Y
     -11070.60+172.3449*T-31.38*T*LN(T)+4.70657E+26*T**(-9);           1700 N !
 $FUNCTION GZNLIQ     298.0  -1.285170+108.1769*T-23.70131*T*LN(T)
 $    -.001712034*T**2-1.264963E-06*T**3-3.585652E-19*T**7;            692.7 Y
 $    -11070.60+172.3449*T-31.38*T*LN(T)+4.70657E+26*T**(-9);           1700 N !
 FUNCTION GZNLIQ     298.14  +7157.213-10.29299*T-3.5896E-19*T**7+GHSERZN#;
                                                                      692.7 Y
     +7450.168-10.737066*T-4.7051E+26*T**(-9)+GHSERZN#;                 1700 N !
 FUNCTION GZNFCC     298.15  +2969.82-1.56968*T+GHSERZN#;               1700 N !

$-------------------------------------------------------------------------------
 TYPE_DEFINITION % SEQ *!
 DEFINE_SYSTEM_DEFAULT ELEMENT 2 !
 DEFAULT_COMMAND DEF_SYS_ELEMENT VA /- !

$-------------------------------------------------------------------------------
$ PARAMETERS FOR LIQUID PHASE
$-------------------------------------------------------------------------------
 PHASE LIQUID %  1  1.0  !
 CONSTITUENT LIQUID :AL,ZN :  !
   PARAMETER G(LIQUID,AL;0)      298.15  +GALLIQ#;                      2900 N !
   PARAMETER G(LIQUID,ZN;0)      298.15  +GZNLIQ#;                      1700 N !
   PARAMETER G(LIQUID,AL,ZN;0)   298.15  +10465.5-3.39259*T;            6000 N !

$-------------------------------------------------------------------------------
$ FUNCTIONS FOR FCC_A1
$-------------------------------------------------------------------------------
 PHASE FCC_A1  %  1  1.0  !
 CONSTITUENT FCC_A1  :AL,ZN :  !
   PARAMETER G(FCC_A1,AL;0)      298.15  +GHSERAL#;                     2900 N !
   PARAMETER G(FCC_A1,ZN;0)      298.15  +GZNFCC#;                      1700 N !
   PARAMETER G(FCC_A1,AL,ZN;0)   298.15  +7297.5+0.47512*T;             6000 N !
   PARAMETER G(FCC_A1,AL,ZN;1)   298.15  +6612.9-4.5911*T;              6000 N !
   PARAMETER G(FCC_A1,AL,ZN;2)   298.15  -3097.2+3.30635*T;             6000 N !

$-------------------------------------------------------------------------------
$ FUNCTIONS FOR HCP_A3
$-------------------------------------------------------------------------------
 PHASE HCP_A3  %  1  1.0  !
 CONSTITUENT HCP_A3  :AL,ZN :  !
  PARAMETER G(HCP_A3,AL;0)       298.15  +GALHCP#;                      2900 N !
  PARAMETER G(HCP_A3,ZN;0)       298.15  +GHSERZN#;                     1700 N !
  PARAMETER G(HCP_A3,AL,ZN;0)    298.15  +18821.0-8.95255*T;            6000 N !
  PARAMETER G(HCP_A3,AL,ZN;3)    298.15  -702.8;                        6000 N !

$
$------------------------------------------------------------------- END OF LINE

"""

dbf      = Database(ALZN_TDB)
comps    = ['AL', 'ZN', 'VA']
phases   = list(dbf.phases.keys())   # ['LIQUID','FCC_A1','HCP_A3']
SEL      = 'ZN'          # element on the composition axis
T_MIN, T_MAX = 500, 1100
T_INIT   = 800
STATE    = {}             # shared state for cross-cell updates
print("Database loaded  |  phases:", phases)


In [ ]:
# @title Step 4 — Common-tangent helper functions

def _tangents_liq_sol(x_liq, g_liq, x_sol, g_sol):
    """Return list of (x1,g1,x2,g2) for liquid<->solid common tangents."""
    dx = 1e-4
    m1 = np.isfinite(g_liq);  xl, gl = x_liq[m1], g_liq[m1]
    m2 = np.isfinite(g_sol);  xs, gs = x_sol[m2], g_sol[m2]
    if len(xl) < 4 or len(xs) < 4:
        return []
    il = interp1d(xl, gl, kind="cubic", bounds_error=False, fill_value=np.nan)
    ir = interp1d(xs, gs, kind="cubic", bounds_error=False, fill_value=np.nan)

    def obj(p):
        a, b = p
        if b - a < 0.02:
            return 1e10
        va, vb = il(a), ir(b)
        if np.isnan(va) or np.isnan(vb):
            return 1e10
        sec = (vb - va) / (b - a + 1e-12)
        da  = (il(a + dx) - il(a - dx)) / (2 * dx)
        db  = (ir(b + dx) - ir(b - dx)) / (2 * dx)
        return (da - sec)**2 + (db - sec)**2

    seen, out = [], []
    for a0 in np.linspace(xl.min() + 0.05, xl.max() - 0.05, 10):
        for b0 in np.linspace(xs.min() + 0.05, xs.max() - 0.05, 10):
            if b0 <= a0:
                continue
            r = minimize(obj, [a0, b0], method="Nelder-Mead",
                         options={"xatol": 1e-5, "fatol": 1e-5, "maxiter": 800})
            if r.fun > 1e-3:
                continue
            a, b = r.x
            if not (xl.min() < a < xl.max() and xs.min() < b < xs.max()):
                continue
            if b - a < 0.02:
                continue
            if any(abs(a - s[0]) < 0.04 and abs(b - s[1]) < 0.04 for s in seen):
                continue
            ga_r, gb_r = float(il(a)), float(ir(b))
            if np.isfinite(ga_r) and np.isfinite(gb_r):
                seen.append((a, b))
                out.append((a, ga_r, b, gb_r))
    return out


def _miscibility_tangent(xf, gf):
    """Common tangent inside a single-phase miscibility gap (FCC)."""
    mask = np.isfinite(gf);  xf, gf = xf[mask], gf[mask]
    if len(xf) < 6:
        return None
    dx  = 1e-4
    ip  = interp1d(xf, gf, kind="cubic", bounds_error=False, fill_value=np.nan)

    def obj(p):
        a, b = p
        if a <= 0.01 or b >= 0.99 or b - a < 0.05:
            return 1e10
        va, vb = ip(a), ip(b)
        if np.isnan(va) or np.isnan(vb):
            return 1e10
        sec = (vb - va) / (b - a)
        da  = (ip(a + dx) - ip(a - dx)) / (2 * dx)
        db  = (ip(b + dx) - ip(b - dx)) / (2 * dx)
        return (da - sec)**2 + (db - sec)**2 + (da - db)**2

    best = None
    for a0 in np.linspace(0.05, 0.45, 6):
        for b0 in np.linspace(0.55, 0.95, 6):
            r = minimize(obj, [a0, b0], method="Nelder-Mead",
                         options={"xatol": 1e-5, "fatol": 1e-5, "maxiter": 1500})
            if r.fun < 1e-3:
                a, b = r.x
                if 0 < a < b < 1 and b - a > 0.05:
                    va, vb = float(ip(a)), float(ip(b))
                    if np.isfinite(va) and np.isfinite(vb):
                        if best is None or r.fun < best[4]:
                            best = (a, va, b, vb, r.fun)
    return best[:4] if best else None

print("Helper functions ready")


## 1. Binary Phase Diagram

The **dashed gold line** tracks the temperature you select in the G–x section below.


In [ ]:
# @title Step 5 — Phase Diagram (computed once; may take ~30 s)
print("Computing phase diagram … please wait.")
plt.close("all")
try:
    binplot(dbf, comps, phases,
            {v.X(SEL): (0, 1, 0.02), v.T: (T_MIN, T_MAX, 10), v.P: 101325})
    _fig_pd = plt.gcf()
    _ax_pd  = plt.gca()
except Exception as e:
    _fig_pd, _ax_pd = plt.subplots(figsize=(8, 5))
    _ax_pd.text(0.5, 0.5, f"binplot error:\n{e}", ha="center", va="center",
                transform=_ax_pd.transAxes, color="red")
_ax_pd.set_title("Al-Zn Phase Diagram (Mey 1993)", fontweight="bold")
_ax_pd.set_xlabel(f"Mole Fraction {SEL}")
_ax_pd.set_ylabel("Temperature (K)")
_ax_pd.set_ylim(T_MIN, T_MAX)
_fig_pd.tight_layout()

# Store figure/axis so the G-x slider can update the temperature line
STATE["fig_pd"] = _fig_pd
STATE["ax_pd"]  = _ax_pd
STATE["tline"]  = None
_pd_out = widgets.Output()
STATE["pd_out"] = _pd_out

def _refresh_pd(T):
    """Update the gold temperature line on the phase diagram and redisplay."""
    ax  = STATE["ax_pd"]
    fig = STATE["fig_pd"]
    if STATE["tline"] is not None:
        try:
            STATE["tline"].remove()
        except Exception:
            pass
    STATE["tline"] = ax.axhline(y=T, color="gold", lw=2.5, ls="--",
                                label=f"T = {T:.0f} K", zorder=10)
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[-2:], labels[-2:], fontsize=8, loc="upper right")
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=110)
    buf.seek(0)
    with STATE["pd_out"]:
        clear_output(wait=True)
        display(IPImage(buf.read()))

_refresh_pd(T_INIT)
display(_pd_out)
print("Phase diagram ready!")


## 2. Gibbs Free Energy Curves  —  Common Tangent Construction

Phase coexistence occurs when the chemical potentials of all components are equal
across the coexisting phases. Graphically this is the **common tangent** (dashed
black line; contact points shown as filled circles).

Move the slider to change temperature and watch how:
- the curves shift
- the common tangent changes endpoints (equilibrium compositions)
- the gold line on the phase diagram above tracks your selection


In [ ]:
# @title Step 6 — Interactive G-x plot with common tangents
_PH_COL = {
    "LIQUID": "#2196F3", "FCC_A1": "#F44336",
    "HCP_A3": "#4CAF50", "BCC_A2": "#FF9800", "HCP_ZN": "#9C27B0",
}
_ge_out = widgets.Output()

def _draw_ge(T):
    # Build all plot data FIRST, then render inside Output context
    xd, gd, cd = {}, {}, {}
    _default = plt.cm.tab10.colors
    ci = 0
    errors = []
    ph_data = []   # list of (ph, xv, gv, col)

    for ph in phases:
        try:
            r  = calculate(dbf, comps, ph, P=101325, T=T, output="GM")
            xv = r.X.sel(component=SEL).values.flatten()
            gv = r.GM.values.flatten()
            idx = np.argsort(xv);  xv, gv = xv[idx], gv[idx]
            mask = np.isfinite(gv)
            if mask.sum() < 3:
                continue
            col = _PH_COL.get(ph, _default[ci % len(_default)])
            xd[ph] = xv;  gd[ph] = gv;  cd[ph] = col
            ph_data.append((ph, xv, gv, mask, col))
            ci += 1
        except Exception as e:
            errors.append(f"{ph}: {e}")

    # Compute common tangents before entering widget context
    tangent_lines = []
    liq = [p for p in xd if "LIQUID" in p.upper()]
    sol = [p for p in xd if "LIQUID" not in p.upper()]
    for lp in liq:
        for sp in sol:
            for pts in _tangents_liq_sol(xd[lp], gd[lp], xd[sp], gd[sp]):
                tangent_lines.append(("ls", pts))
    for sp in sol:
        res = _miscibility_tangent(xd[sp], gd[sp])
        if res:
            tangent_lines.append(("ph", res, cd[sp]))

    # Now render everything inside the Output widget
    with _ge_out:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(9, 5))
        for ph, xv, gv, mask, col in ph_data:
            ax.plot(xv[mask], gv[mask], color=col, label=ph, zorder=3)
        if errors and not ph_data:
            ax.text(0.5, 0.5, "Errors:
" + "
".join(errors),
                    ha="center", va="center", transform=ax.transAxes,
                    color="red", fontsize=8)
        first = True
        for item in tangent_lines:
            if item[0] == "ls":
                x1, g1, x2, g2 = item[1]
                ax.plot([x1, x2], [g1, g2], "k--", lw=1.8, zorder=5,
                        label="Common tangent" if first else "")
                ax.plot([x1, x2], [g1, g2], "ko", ms=7, zorder=6)
                first = False
            else:
                x1, g1, x2, g2 = item[1]
                col = item[2]
                ax.plot([x1, x2], [g1, g2], color=col, ls="--", lw=1.8, zorder=5)
                ax.plot([x1, x2], [g1, g2], "o", color=col, ms=7, zorder=6)
        ax.set_title(f"Molar Gibbs Free Energy  —  T = {T:.0f} K", fontweight="bold")
        ax.set_xlabel(f"Mole Fraction {SEL}")
        ax.set_ylabel(r"$G_m$ (J/mol)")
        ax.set_xlim(-0.02, 1.02)
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.25)
        plt.tight_layout()
        plt.show()
    _refresh_pd(T)

_sl_ge = widgets.FloatSlider(
    value=T_INIT, min=T_MIN, max=T_MAX, step=25,
    description="T (K):", continuous_update=False,
    layout=widgets.Layout(width="70%"))
_sl_ge.observe(lambda c: _draw_ge(c["new"]), names="value")
# Fill the Output widget BEFORE displaying so Colab renders it immediately
_draw_ge(T_INIT)
display(_sl_ge, _ge_out)


## 3. Thermodynamic Properties

Enthalpy, Entropy and Activity of Zn vs. composition.
Each phase is shown separately with its phase-diagram colour.


In [ ]:
# @title Step 7 — Interactive thermodynamic properties
_tp_out = widgets.Output()

def _draw_tp(T):
    # Collect all data before entering Output context
    hm_data, sm_data = [], []
    _default = plt.cm.tab10.colors
    ci = 0
    hm_errs, sm_errs = [], []

    for ph in phases:
        col = _PH_COL.get(ph, _default[ci % len(_default)])
        ci += 1
        try:
            rh = calculate(dbf, comps, ph, P=101325, T=T, output="HM")
            xh = rh.X.sel(component=SEL).values.flatten()
            hm = rh.HM.values.flatten()
            idx = np.argsort(xh);  xh, hm = xh[idx], hm[idx]
            hm_data.append((xh, hm, col, ph))
        except Exception as e:
            hm_errs.append(f"{ph}: {e}")
        try:
            rs = calculate(dbf, comps, ph, P=101325, T=T, output="SM")
            xs = rs.X.sel(component=SEL).values.flatten()
            sm = rs.SM.values.flatten()
            idx = np.argsort(xs);  xs, sm = xs[idx], sm[idx]
            sm_data.append((xs, sm, col, ph))
        except Exception as e:
            sm_errs.append(f"{ph}: {e}")

    act_data, act_err = None, None
    try:
        conds  = {v.X(SEL): (0.01, 0.99, 0.02), v.T: T, v.P: 101325}
        eq     = equilibrium(dbf, comps, phases, conds)
        xeq    = eq.X.sel(component=SEL).squeeze().values.flatten()
        mu     = eq.MU.sel(component=SEL).squeeze().values.flatten()
        n      = min(len(xeq), len(mu));  xeq, mu = xeq[:n], mu[:n]
        ref    = equilibrium(dbf, comps, ["HCP_A3"],
                             {v.X(SEL): 0.9999, v.T: T, v.P: 101325})
        mu_ref = float(ref.MU.sel(component=SEL).values.flatten()[0])
        act    = np.exp((mu - mu_ref) / (8.314 * T))
        idx    = np.argsort(xeq);  xeq, act = xeq[idx], act[idx]
        act_data = (xeq, act)
    except Exception as e:
        act_err = str(e)

    # Render inside Output context
    with _tp_out:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        fig.suptitle(f"Thermodynamic Properties — T = {T:.0f} K", fontweight="bold")
        for xh, hm, col, ph in hm_data:
            axes[0].plot(xh[np.isfinite(hm)], hm[np.isfinite(hm)], color=col, label=ph)
        for i, msg in enumerate(hm_errs):
            axes[0].text(0.5, 0.5 - i*0.12, msg, transform=axes[0].transAxes,
                         color="red", fontsize=7, ha="center", va="center")
        for xs, sm, col, ph in sm_data:
            axes[1].plot(xs[np.isfinite(sm)], sm[np.isfinite(sm)], color=col, label=ph)
        for i, msg in enumerate(sm_errs):
            axes[1].text(0.5, 0.5 - i*0.12, msg, transform=axes[1].transAxes,
                         color="red", fontsize=7, ha="center", va="center")
        if act_data is not None:
            xeq, act = act_data
            mask = np.isfinite(act)
            axes[2].plot(xeq[mask], act[mask], color="#FF5722", label="Calculated")
            axes[2].plot([0, 1], [0, 1], "k--", label="Raoult's Law")
            axes[2].set_ylim(-0.05, 1.05)
        else:
            axes[2].text(0.5, 0.5, f"Activity error:
{act_err}",
                         transform=axes[2].transAxes, color="red", fontsize=9,
                         ha="center", va="center")
        titles  = ["Enthalpy of Mixing", "Entropy of Mixing", f"Activity of {SEL}"]
        ylabels = [r"$H_m$ (J/mol)", r"$S_m$ (J/K$\cdot$mol)", f"$a_{{\mathrm{{{SEL}}}}}$"]
        for ax, ti, yl in zip(axes, titles, ylabels):
            ax.set_title(ti, fontweight="bold")
            ax.set_xlabel(f"$X_{{\mathrm{{{SEL}}}}}$")
            ax.set_ylabel(yl)
            ax.grid(True, alpha=0.25)
            ax.legend(fontsize=8)
        plt.tight_layout()
        plt.show()

_sl_tp = widgets.FloatSlider(
    value=T_INIT, min=T_MIN, max=T_MAX, step=50,
    description="T (K):", continuous_update=False,
    layout=widgets.Layout(width="70%"))
_sl_tp.observe(lambda c: _draw_tp(c["new"]), names="value")
# Fill the Output widget BEFORE displaying so Colab renders it immediately
_draw_tp(T_INIT)
display(_sl_tp, _tp_out)
